**Install Required Packages**

In [1]:
!pip install -U peft bitsandbytes transformers accelerate

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 13.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.7/10.7 MB 79.6 MB/s eta 0:00:00
  Attempting uninstall: transformers
    Found existing installation: transformers 5.0.0
    Uninstalling transformers-5.0.0:
      Successfully uninstalled transformers-5.0.0


In [2]:
!pip install -U trl

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 531.0/531.0 kB 37.3 MB/s eta 0:00:00


In [3]:
!pip install PyMuPDF

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.9/24.9 MB 57.4 MB/s eta 0:00:00


**PreBuilt Data from HuggingFace data Hub**

In [4]:
from datasets import Dataset, load_dataset

In [5]:
dataset = load_dataset("roneneldan/TinyStories", split="train")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

data/train-00000-of-00004-2d5a1467fff108(…):   0%|          | 0.00/249M [00:00<?, ?B/s]

data/train-00001-of-00004-5852b56a2bd28f(…):   0%|          | 0.00/248M [00:00<?, ?B/s]

data/train-00002-of-00004-a26307300439e9(…):   0%|          | 0.00/246M [00:00<?, ?B/s]

data/train-00003-of-00004-d243063613e5a0(…):   0%|          | 0.00/248M [00:00<?, ?B/s]

data/validation-00000-of-00001-869c898b5(…):   0%|          | 0.00/9.99M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/2119719 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/21990 [00:00<?, ? examples/s]

In [6]:
print(dataset)

Dataset({
    features: ['text'],
    num_rows: 2119719
})


**Own Custom Data for Domain Specific Fine Tuning**

In [7]:
import fitz

In [8]:
def extract_text_from_pdf(pdf_path):
  text_blocks = []
  with fitz.open(pdf_path) as doc:
    for page in doc:
      text = page.get_text("text").strip()
      if next:
        text_blocks.append(text)
  return text_blocks

In [9]:
pdf_texts = extract_text_from_pdf("/content/Metformin.pdf")

In [10]:
pdf_texts

['Metformin is an oral medication widely used to manage type 2 diabetes mellitus. It belongs \nto a class of drugs called biguanides. It is considered a first-line treatment due to its \neffectiveness, safety profile, and low cost. \n \nMechanism of Action \nMetformin works primarily by reducing glucose production in the liver (hepatic \ngluconeogenesis). It also improves insulin sensitivity in peripheral tissues such as muscles \nand fat cells. \nKey actions: \n• \nDecreases hepatic glucose output \n• \nIncreases insulin sensitivity \n• \nEnhances peripheral glucose uptake \n• \nReduces intestinal glucose absorption \nMetformin does not stimulate insulin secretion, which means it has a low risk of causing \nhypoglycemia. \n \nIndications \nMetformin is prescribed for: \n• \nType 2 diabetes mellitus \n• \nPrediabetes (to delay progression to diabetes) \n• \nPolycystic ovary syndrome (PCOS) (off-label use) \n• \nGestational diabetes (in some cases) \n \nDosage and Administration \nMetfo

In [11]:
import re
def split_paragraphs(pages):
  paragraphs = []
  for page_text in pages:
    #Split on Double line breaks or long newlines
    chunks = re.split(r'\n\s*\n', page_text)
    for chunk in chunks:
      clean = chunk.strip()
      if len(clean) > 30:
        paragraphs.append(clean)
  return paragraphs

In [12]:
paragraphs = split_paragraphs(pdf_texts)

In [13]:
paragraphs

['Metformin is an oral medication widely used to manage type 2 diabetes mellitus. It belongs \nto a class of drugs called biguanides. It is considered a first-line treatment due to its \neffectiveness, safety profile, and low cost.',
 'Mechanism of Action \nMetformin works primarily by reducing glucose production in the liver (hepatic \ngluconeogenesis). It also improves insulin sensitivity in peripheral tissues such as muscles \nand fat cells. \nKey actions: \n• \nDecreases hepatic glucose output \n• \nIncreases insulin sensitivity \n• \nEnhances peripheral glucose uptake \n• \nReduces intestinal glucose absorption \nMetformin does not stimulate insulin secretion, which means it has a low risk of causing \nhypoglycemia.',
 'Indications \nMetformin is prescribed for: \n• \nType 2 diabetes mellitus \n• \nPrediabetes (to delay progression to diabetes) \n• \nPolycystic ovary syndrome (PCOS) (off-label use) \n• \nGestational diabetes (in some cases)',
 'Dosage and Administration \nMetformi

In [14]:
data = [{"text" : p} for p in paragraphs]

In [15]:
data

[{'text': 'Metformin is an oral medication widely used to manage type 2 diabetes mellitus. It belongs \nto a class of drugs called biguanides. It is considered a first-line treatment due to its \neffectiveness, safety profile, and low cost.'},
 {'text': 'Mechanism of Action \nMetformin works primarily by reducing glucose production in the liver (hepatic \ngluconeogenesis). It also improves insulin sensitivity in peripheral tissues such as muscles \nand fat cells. \nKey actions: \n• \nDecreases hepatic glucose output \n• \nIncreases insulin sensitivity \n• \nEnhances peripheral glucose uptake \n• \nReduces intestinal glucose absorption \nMetformin does not stimulate insulin secretion, which means it has a low risk of causing \nhypoglycemia.'},
 {'text': 'Indications \nMetformin is prescribed for: \n• \nType 2 diabetes mellitus \n• \nPrediabetes (to delay progression to diabetes) \n• \nPolycystic ovary syndrome (PCOS) (off-label use) \n• \nGestational diabetes (in some cases)'},
 {'text'

In [16]:
dataset = Dataset.from_list(data)

In [17]:
dataset

Dataset({
    features: ['text'],
    num_rows: 12
})

In [18]:
dataset[0]

{'text': 'Metformin is an oral medication widely used to manage type 2 diabetes mellitus. It belongs \nto a class of drugs called biguanides. It is considered a first-line treatment due to its \neffectiveness, safety profile, and low cost.'}

**Let Select the Model**

In [19]:
model_name = "TinyLlama/TinyLlama-1.1B-intermediate-step-1431k-3T"

In [20]:
from transformers import AutoTokenizer, AutoModelForCausalLM, Trainer, TrainingArguments, DataCollatorForLanguageModeling

In [21]:
tokenizer = AutoTokenizer.from_pretrained(model_name)

config.json:   0%|          | 0.00/560 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/776 [00:00<?, ?B/s]

tokenizer.model:   0%|          | 0.00/500k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/414 [00:00<?, ?B/s]

In [22]:
if tokenizer.pad_token_id is None:
  tokenizer.pad_token = tokenizer.eos_token

**Pre-Process Data**

In [23]:
def tokenize_fn(examples):
  tokens = tokenizer(examples["text"], truncation=True, padding="max_length",max_length=512)
  tokens["labels"] = tokens["input_ids"].copy()
  return tokens

In [24]:
tokenized = dataset.map(tokenize_fn, batched=True, remove_columns=["text"])

Map:   0%|          | 0/12 [00:00<?, ? examples/s]

In [25]:
tokenized

Dataset({
    features: ['input_ids', 'attention_mask', 'labels'],
    num_rows: 12
})

In [26]:
model = AutoModelForCausalLM.from_pretrained(model_name)

model.safetensors:   0%|          | 0.00/4.40G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/129 [00:00<?, ?B/s]

**Training Arugments**

In [27]:
pip install --upgrade transformers

In [28]:
training_args = TrainingArguments(
    output_dir="./llama-pharma-domain",
    num_train_epochs=2,
    per_device_train_batch_size=2,
    save_steps=500,
    save_total_limit=2,
    logging_steps=10,
    learning_rate=2e-5,
    fp16=True,
    report_to="none"
)

In [29]:
from transformers import TrainingArguments
help(TrainingArguments)

Help on class TrainingArguments in module transformers.training_args:

class TrainingArguments(builtins.object)
 |  TrainingArguments(output_dir: str | None = None, per_device_train_batch_size: int = 8, num_train_epochs: float = 3.0, max_steps: int = -1, learning_rate: float = 5e-05, lr_scheduler_type: transformers.trainer_utils.SchedulerType | str = 'linear', lr_scheduler_kwargs: dict | str | None = None, warmup_steps: float = 0, optim: transformers.training_args.OptimizerNames | str = 'adamw_torch_fused', optim_args: str | None = None, weight_decay: float = 0.0, adam_beta1: float = 0.9, adam_beta2: float = 0.999, adam_epsilon: float = 1e-08, optim_target_modules: None | str | list[str] = None, gradient_accumulation_steps: int = 1, average_tokens_across_devices: bool = True, max_grad_norm: float = 1.0, label_smoothing_factor: float = 0.0, bf16: bool = False, fp16: bool = False, bf16_full_eval: bool = False, fp16_full_eval: bool = False, tf32: bool | None = None, gradient_checkpointing

In [30]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized
    )

**LoRA Based Method**

In [31]:
import torch, gc
gc.collect()
torch.cuda.empty_cache()

In [32]:
!pip install -U bitsandbytes transformers accelerate

In [33]:
from transformers import AutoTokenizer, AutoModelForCausalLM, TrainingArguments, Trainer
from peft import LoraConfig, get_peft_model, TaskType
from datasets import load_dataset

In [34]:
model = "TinyLlama/TinyLlama-1.1B-intermediate-step-1431k-3T"

In [35]:
tokenizer = AutoTokenizer.from_pretrained(model)

In [36]:
if tokenizer.pad_token is None:
  tokenizer.pad_token = tokenizer.eos_token

In [37]:
def tokenize_fn(examples):
  tokens = tokenizer(examples["text"],
                     truncation=True,
                     padding="max_length",
                     max_length=512)
  tokens["labels"] = tokens["input_ids"].copy()
  return tokens

In [38]:
tokenized = dataset.map(tokenize_fn, batched=True)

Map:   0%|          | 0/12 [00:00<?, ? examples/s]

In [39]:
tokenized

Dataset({
    features: ['text', 'input_ids', 'attention_mask', 'labels'],
    num_rows: 12
})

In [40]:
pip install -U transformers accelerate bitsandbytes

In [41]:
from transformers import BitsAndBytesConfig

bnb_config = BitsAndBytesConfig(
    load_in_8bit=True
)

model = AutoModelForCausalLM.from_pretrained(model_name)

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

In [42]:
lora_config = LoraConfig(
    task_type=TaskType.CAUSAL_LM,
    r=8,
    lora_alpha=16,
    target_modules=["q_proj", "v_proj"],
    lora_dropout=0.05,
    bias="none",
)

In [43]:
q_lora_model = get_peft_model(model, lora_config)

In [44]:
args = TrainingArguments(
    output_dir="./tinyllama-lora",
    num_train_epochs=5,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=8,
    learning_rate=2e-4,
    fp16=False,
    logging_steps=20,
    save_total_limit=1,
    report_to="none"
)

In [45]:
model

LlamaForCausalLM(
  (model): LlamaModel(
    (embed_tokens): Embedding(32000, 2048)
    (layers): ModuleList(
      (0-21): 22 x LlamaDecoderLayer(
        (self_attn): LlamaAttention(
          (q_proj): lora.Linear(
            (base_layer): Linear(in_features=2048, out_features=2048, bias=False)
            (lora_dropout): ModuleDict(
              (default): Dropout(p=0.05, inplace=False)
            )
            (lora_A): ModuleDict(
              (default): Linear(in_features=2048, out_features=8, bias=False)
            )
            (lora_B): ModuleDict(
              (default): Linear(in_features=8, out_features=2048, bias=False)
            )
            (lora_embedding_A): ParameterDict()
            (lora_embedding_B): ParameterDict()
            (lora_magnitude_vector): ModuleDict()
          )
          (k_proj): Linear(in_features=2048, out_features=256, bias=False)
          (v_proj): lora.Linear(
            (base_layer): Linear(in_features=2048, out_features=256, bia

In [46]:
trainer = Trainer(
    model=q_lora_model,
    args=args,
    train_dataset=tokenized
)

In [47]:
trainer.train()

Step,Training Loss


TrainOutput(global_step=10, training_loss=10.133753967285156, metrics={'train_runtime': 44.4351, 'train_samples_per_second': 1.35, 'train_steps_per_second': 0.225, 'total_flos': 190888940666880.0, 'train_loss': 10.133753967285156, 'epoch': 5.0})

In [48]:
model_path = "/content/tinyllama-lora/checkpoint-10"

In [49]:
trained_model = AutoModelForCausalLM.from_pretrained(model_path, device_map="auto")

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/88 [00:00<?, ?it/s]

In [50]:
prompt = "Metformin does not stimulate insulin secretion, which means it has a low risk of causing hypoglycemia."

In [51]:
inputs = tokenizer(prompt, return_tensors="pt").to("cuda")

In [52]:
outputs = trained_model.generate(
    **inputs,
    max_new_tokens=100,
    temperature=0.8,
    top_p=0.9,
    do_sample=True,
    repetition_penalty=1.1
)

Both `max_new_tokens` (=100) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


In [53]:
print("\nModel Output: \n")
print(tokenizer.decode(outputs[0], skip_special_tokens=True))


Model Output: 

Metformin does not stimulate insulin secretion, which means it has a low risk of causing hypoglycemia. It also doesn't affect glycogenolysis and doesn't increase glucose production by the liver.
Metformin has no known side effects, though some users have reported nausea, diarrhea, gas, bloating, and constipation after taking this medication. Metformin is contraindicated in patients with a history of peptic ulcers or hemorrhagic stroke. Metformin can cause dizziness,
